# 03 Error Analysis

## Purpose

This notebook performs syllable-level error diagnostics for a selected `my-ph` experiment. It classifies errors into ambiguity, OOV, missing-reference, and over-generalization categories.

In [1]:
from pathlib import Path
import subprocess, re, shutil, os, difflib

BASE_DIR = Path('/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein')
G2P_DIR = BASE_DIR / 'g2p-par'
SGM_DIR = G2P_DIR / 'sgm'
MOSES_SRC_DIR = Path('/home/kyalkalay/tools/mosesdecoder')
MOSES_SCRIPT_DIR = MOSES_SRC_DIR / 'scripts'
MOSES_BIN_DIR = MOSES_SRC_DIR / 'bin'
EXPERIMENT_PERL = MOSES_SCRIPT_DIR / 'ems' / 'experiment.perl'
MTEVAL = MOSES_SCRIPT_DIR / 'generic' / 'mteval-v13a.pl'

# The original assignment experiment directory.
ORIGINAL_EXP_DIR = BASE_DIR / 'pbsmt-big-normalize'
ORIGINAL_TEMPLATE = ORIGINAL_EXP_DIR / 'config.baseline'
ORIGINAL_BASELINE_DIR = ORIGINAL_EXP_DIR / 'baseline'

# New clean experiment directory.
EXP_DIR = BASE_DIR / 'pbsmt_original_clean_experiments'

print('BASE_DIR:', BASE_DIR)
print('G2P_DIR:', G2P_DIR)
print('SGM_DIR:', SGM_DIR)
print('Original template:', ORIGINAL_TEMPLATE, ORIGINAL_TEMPLATE.exists())
print('Clean experiment dir:', EXP_DIR)

BASE_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein
G2P_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par
SGM_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/sgm
Original template: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt-big-normalize/config.baseline True
Clean experiment dir: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments


In [2]:
import pandas as pd
from collections import defaultdict, Counter

EXPERIMENT = 'pbsmt_5gram_main'
DIRECTION = 'my-ph'
EVAL_DIR = EXP_DIR / EXPERIMENT / DIRECTION / 'evaluation'

file_train_src = G2P_DIR / 'train_clean.my'
file_train_trg = G2P_DIR / 'train_clean.ph'
file_test_src  = EVAL_DIR / 'test.input.txt.1'
file_test_out  = EVAL_DIR / 'test.output.1'
file_test_ref  = EVAL_DIR / 'test.reference.txt.1'

for p in [file_train_src, file_train_trg, file_test_src, file_test_out, file_test_ref]:
    print(p, p.exists())

/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/train_clean.my True
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/train_clean.ph True
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph/evaluation/test.input.txt.1 True
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph/evaluation/test.output.1 True
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments/pbsmt_5gram_main/my-ph/evaluation/test.reference.txt.1 True


In [3]:
def clean_moses_output(line):
    return ' '.join(re.sub(r'\|\d+-\d+\|', '', line).split()).strip()

train_counts = defaultdict(Counter)
with file_train_src.open(encoding='utf-8') as f_src, file_train_trg.open(encoding='utf-8') as f_trg:
    for src_line, trg_line in zip(f_src, f_trg):
        s_tokens = src_line.strip().split()
        t_tokens = trg_line.strip().split()
        if len(s_tokens) == len(t_tokens):
            for s, t in zip(s_tokens, t_tokens):
                train_counts[s][t] += 1

raw_errors = []
with file_test_src.open(encoding='utf-8') as f_src, file_test_out.open(encoding='utf-8') as f_out, file_test_ref.open(encoding='utf-8') as f_ref:
    for src, out, ref in zip(f_src, f_out, f_ref):
        src_tokens = src.strip().split()
        ref_tokens = ref.strip().split()
        out_line = out.strip()
        tagged = re.findall(r'([^|]+?)\|(\d+)-(\d+)\|', out_line)
        covered = set()
        if tagged:
            for pred_phrase, start_s, end_s in tagged:
                start, end = int(start_s), int(end_s)
                pred = ' '.join(pred_phrase.split()).strip()
                if end < len(ref_tokens):
                    src_chunk = ' '.join(src_tokens[start:end+1])
                    ref_chunk = ' '.join(ref_tokens[start:end+1])
                    covered.update(range(start, end+1))
                    if pred != ref_chunk:
                        raw_errors.append((src_chunk, pred, ref_chunk))
            for i in range(min(len(src_tokens), len(ref_tokens))):
                if i not in covered:
                    raw_errors.append((src_tokens[i], '[DELETED]', ref_tokens[i]))
        else:
            pred_tokens = clean_moses_output(out_line).split()
            for s, pred, r in zip(src_tokens, pred_tokens, ref_tokens):
                if pred != r:
                    raw_errors.append((s, pred, r))

records = []
for src_chunk, pred, ref in raw_errors:
    tr = train_counts.get(src_chunk, Counter())
    total = sum(tr.values())
    pred_freq = tr.get(pred, 0)
    ref_freq = tr.get(ref, 0)
    if total == 0:
        cause = 'Out of Vocabulary (OOV)'
    elif pred_freq > 0 and ref_freq > 0:
        cause = 'Train Data Ambiguity (Both Exist)'
    elif ref_freq == 0 and pred_freq > 0:
        cause = 'Reference Missing from Training Token'
    elif pred_freq == 0 and ref_freq > 0:
        cause = 'Decoder Hallucination / Over-generalization'
    else:
        cause = 'Unclassified Structural Swap'
    records.append({
        'Source_Syllable': src_chunk,
        'Moses_Output': pred,
        'True_Reference': ref,
        'Train_Total_Seen': total,
        'Train_Prob_Pred': round(pred_freq/total, 3) if total else 0.0,
        'Train_Prob_Ref': round(ref_freq/total, 3) if total else 0.0,
        'Root_Cause': cause,
    })

df_errors = pd.DataFrame(records)
summary = df_errors['Root_Cause'].value_counts().rename_axis('Root_Cause').reset_index(name='Count') if not df_errors.empty else pd.DataFrame()
if not summary.empty:
    summary['Percentage'] = (summary['Count'] / summary['Count'].sum() * 100).round(2)
leaderboard = df_errors.groupby(['Source_Syllable','Moses_Output','True_Reference','Root_Cause']).size().reset_index(name='Error_Count').sort_values('Error_Count', ascending=False) if not df_errors.empty else pd.DataFrame()

df_errors.to_csv(f'error_details_{EXPERIMENT}_{DIRECTION}.csv', index=False)
summary.to_csv(f'error_summary_{EXPERIMENT}_{DIRECTION}.csv', index=False)
leaderboard.to_csv(f'error_leaderboard_{EXPERIMENT}_{DIRECTION}.csv', index=False)
summary

,Root_Cause,Count,Percentage
0,Train Data Ambiguity (Both Exist),856,72.91
1,Out of Vocabulary (OOV),232,19.76
2,Reference Missing from Training Token,85,7.24
3,Decoder Hallucination / Over-generalization,1,0.09


In [4]:
leaderboard.head(50)

,Source_Syllable,Moses_Output,True_Reference,Root_Cause,Error_Count
131,ခံ,khan,gan,Train Data Ambiguity (Both Exist),12
585,မ,ma-,ma.,Train Data Ambiguity (Both Exist),10
162,ခြေ,chei,chi,Train Data Ambiguity (Both Exist),8
688,ဝါး,wa:,wa-,Train Data Ambiguity (Both Exist),8
46,ကျ,kya.,gya.,Train Data Ambiguity (Both Exist),7
492,ပေါက်,bau',pau',Train Data Ambiguity (Both Exist),7
78,ကြား,gya:,kya:,Train Data Ambiguity (Both Exist),7
1,က,ka-,ga-,Train Data Ambiguity (Both Exist),7
219,စား,za:,sa:,Train Data Ambiguity (Both Exist),7
363,တောင်,taun,daun,Train Data Ambiguity (Both Exist),6


In [5]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep 'ခံ'

အ ခံ ဆေး	a- khan |0-1| zei: |2-2| 	a- khan hsei: 
အာ မ ခံ ပစ် စည်း	a ma. gan |0-2| pji' si: |3-4| 	a ma. gan pji' si: 
မိုး ရေ ခံ သင် ကန်း	mou: |0-0| jei |1-1| gan |2-2| thin gan: |3-4| 	mou: jei gan thin gan: 
ခံ တပ်	gan |0-0| ta' |1-1| 	khan da' 
သီ ချင်း ခံ	tha- chin: |0-1| khan |2-2| 	tha- chin: gan 
ကြံ့ ကြံ့ ခံ	kyan. |0-0| kyan. |1-1| khan |2-2| 	kyan. gyan. khan 
တာ ဝန် ခံ	ta |0-0| wun |1-1| khan |2-2| 	ta win gan 
ဆွမ်း ခံ	hsun: |0-0| khan |1-1| 	hswan: khan 
ခေါင်း ခံ	gaun: |0-0| khan |1-1| 	gaun: khan 
ပလ် လင် ခံ	pa- lin |0-1| khan |2-2| 	pa- lin khan 
ခံ ချိ	khan gyi. |0-1| 	khan chi. 
စ ကား ပလ် လင် ခံ	za- ga- |0-1| pa- lin |2-3| khan |4-4| 	za- ga: pa- lin khan 
ခြင် ခံ	chin |0-0| khan |1-1| 	chin gan 
အ ခွင့် ထူး ခံ	a- |0-0| khwin. |1-1| du: |2-2| khan |3-3| 	a- khwin. du: gan 
အ တိုင် ပင် ခံ	a- |0-0| tain |1-1| pin |2-2| khan |3-3| 	a- tain bin gan 
ဂုဏ် ခံ	goun |0-0| khan |1-1| 	goun khan 
အ ရှုံး ခံ	a- |0-0| shoun: |1-1| khan |2-2| 	a- shoun: khan 
က တိ ခံ ဝန် ချက်	ga- 

In [6]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep ' မ '

ဥ မ ကွဲ သိုက် မ ပျက်	u. |0-0| ma- |1-1| kwe: |2-2| thai' |3-3| ma- |4-4| pje' |5-5| 	u. ma- kwe: thai' ma- pje' 
မ ဆုတ် မ ဆိုင်း	ma- |0-0| hsou' |1-1| ma- |2-2| hsain: |3-3| 	ma- hsou' ma- hsain: 
အ လိုက် ကမ်း ဆိုး မ သိ	a- |0-0| lai' |1-1| kan: |2-2| hsou: |3-3| ma- thi. |4-5| 	a- lai' kan: zou: ma- thi. 
အာ မ ခံ ပစ် စည်း	a ma. gan |0-2| pji' si: |3-4| 	a ma. gan pji' si: 
ယင် မ မဲ ရိုင်း	jin |0-0| ma- |1-1| me: |2-2| jain: |3-3| 	jin ma. me: jain: 
အင် ဂ သ မ ဥစ် စာ	in: ga- |0-1| tha- ma. |2-3| ou' sa |4-5| 	in ga. tha- ma. ou' sa 
သား ကြီး မ ယား ကြီး	tha: |0-0| gyi: |1-1| ma- ja: |2-3| gyi: |4-4| 	tha: gyi: ma- ja: gyi: 
ကင်း မ လက် မည်း	kin: |0-0| ma- |1-1| le' |2-2| me: |3-3| 	kin: ma. le' me: 
မိန်း မ မှု	mein: ma- |0-1| mhu. |2-2| 	mein: ma. mhu. 
သ မ ပိုင်း	tha- ma. |0-1| bain: |2-2| 	tha- ma. bain: 
သက် မ ချ	the' |0-0| ma- |1-1| cha. |2-2| 	the' ma. cha. 
မ ရှု မ လှ	ma- |0-0| shu. |1-1| ma- |2-2| lha. |3-3| 	ma- shu. ma- lha. 
အန်း မ ရ	an: |0-0| ma- ja. |1-2| 	an: ma- ja. 
ဝမ်း မ

In [7]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep ' က '

ပက် က လာ	pe' |0-0| ga- |1-1| la |2-2| 	pe' ka- la 
ပ က တိ ပ မာ ဏ	ba- ga- di. |0-2| pa- ma |3-4| na. |5-5| 	ba- ga- di. pa- ma na. 
စီး က ရက်	si: |0-0| ka- |1-1| re' |2-2| 	si: ka- re' 
ပြူး က လူး ပျာ က လာ	pju: |0-0| ka- |1-1| lu: |2-2| bja ga- |3-4| la |5-5| 	pju: ga- lu: pja ga- la 
က တိမ်း က ပါး ဖြစ်	ga- dein: ga- |0-2| ba: |3-3| hpji' |4-4| 	ga- dein: ga- ba: hpji' 
ပြ က တေ့	pja. |0-0| ka- |1-1| tei. |2-2| 	bja. ga- dei. 
သက် ရင်း က တိုး	the' |0-0| jin: |1-1| ga- dou: |2-3| 	the' jin: ga- dou: 
က ပျာ က သီ	ga- |0-0| bja |1-1| ka- thi |2-3| 	ga- bja ga- thi 
အာ ဝေ ဏိ က ဒုက် ခ	a |0-0| wei |1-1| ni. |2-2| ka. |3-3| dou' kha. |4-5| 	a wei ni. ka. dou' kha. 
တ က တဲ မှ ပဲ	da- ga- de: |0-2| mha. |3-3| pe: |4-4| 	da- ga- de: mha. be: 
လူ က လေး	lu |0-0| ga- lei: |1-2| 	lu ga- lei: 
က ရိ က ထ	ka- |0-0| ri. |1-1| ka- |2-2| hta. |3-3| 	ka- ji. ka- hta. 
အ က အ ခုန်	a- |0-0| ka- |1-1| a- |2-2| khoun |3-3| 	a- ka. a- khoun 
ဇ န က ဇာတ်	za- na- |0-1| ka. |2-2| za' |3-3| 	za- na- ka za' 
အော် တို က ရေ 

In [8]:
leaderboard[leaderboard.Root_Cause=='Out of Vocabulary (OOV)'].head(30)

,Source_Syllable,Moses_Output,True_Reference,Root_Cause,Error_Count
202,စ ကား,za- ga:,za- ga-,Out of Vocabulary (OOV),3
110,ခ နဲ,kha- ne:,ga- ne:,Out of Vocabulary (OOV),3
704,သက် က ရာဇ်,the' ka- ji',dha- ga- ji',Out of Vocabulary (OOV),2
644,ရို သေ,jou thei,jou dhei,Out of Vocabulary (OOV),2
240,စော် နံ,zo nan,so nan,Out of Vocabulary (OOV),2
201,စ ကား,za- ga-,za- ga:,Out of Vocabulary (OOV),2
190,ငေါ,ငေါ,ngo:,Out of Vocabulary (OOV),2
674,လှန်း,လှန်း,lhan:,Out of Vocabulary (OOV),2
166,ခြေ ချင်း,chi gyin:,chei gyin:,Out of Vocabulary (OOV),2
778,အဲ့,အဲ့,e.,Out of Vocabulary (OOV),2


In [9]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep 'ပစ် စည်း'

အာ မ ခံ ပစ် စည်း	a ma. gan |0-2| pji' si: |3-4| 	a ma. gan pji' si: 
ပစ် စည်း ပစ် ယံ	pji' si: |0-1| pji' |2-2| jan |3-3| 	pji' si: pji' jan 
ပင် ထွက် ပစ် စည်း	pin |0-0| dwe' pji' si: |1-3| 	pin dwe' pji' si: 
အ သွင်း ပစ် စည်း	a- |0-0| thwin: |1-1| pji' si: |2-3| 	a- thwin: pji' si: 
ကုန် ပစ် စည်း	koun |0-0| pji' si: |1-2| 	koun pji si: 


In [10]:
!paste {file_train_src} {file_train_trg} | grep 'ပစ် စည်း'

သင် ချာ ပစ် စည်း	thin cha pji' si:
ဘေး ထွက် ပစ် စည်း	bei: dwe' pji' si:
လေ လွင့် ပစ် စည်း	lei lwin. pji' si:
ပစ် စည်း ပစ် စ ယ	pji' si: pji' sa- ja.
တွင်း ထွက် ပစ် စည်း	twin: dwe' pji' si:
မြေ ထွက် ပစ် စည်း	mjei dwe' pji' si:
ခန်း ဝင် ပစ် စည်း	khan: win pji' si:
ပစ် စည်း လေး ပါး	pji' si: lei: ba:
စစ် လက် နက် ပစ် စည်း	si' le' ne' pji' si:
ပျော် ဝင် ပစ် စည်း	pjo win pji' si:
ကြွေ ရည် သုတ် ပစ် စည်း	kywei jei dhou' pji' si:
အိမ် ထောင် ပစ် စည်း	ein daun pji' si:
ပစ် စည်း	pji' si:
ရှေး ဟောင်း ပစ် စည်း	shei: haun: pji' si:
ဘုံ ပစ် စည်း	boun pji' si:
ကား ပစ် စည်း	ka: pji' si:
စက် ပစ် စည်း	se' pji' si:
ရေ ထွက် ပစ် စည်း	jei dwe' pji' si:
ခိုး ရာ ပါ ပစ် စည်း	khou: ja ba pji' si:
ဇိမ် ခံ ပစ် စည်း	zein gan pji' si:
အ တွင်း ပစ် စည်း	a- twin: pji' si:
သက် သေ ခံ ပစ် စည်း	the' thei gan pji' si:
ပစ် စည်း မဲ့ လူ တန်း စား	pji' si: me: lu dan: za:
ကိုယ် ပိုင် ပစ် စည်း	kou bain pji' si:
ပစ် စည်း ရှိ	pji' si: shi.
ဘိုး ဘ ပိုင် ပစ် စည်း	bou: ba. bain pji' si:


In [11]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep 'အ ဘိ '

အ ဘိ ယာ စ က	a- |0-0| bi. |1-1| ja |2-2| za- ga. |3-4| 	a- bi. ja za- ka. 
အ ဘိ ညာဉ်	a- |0-0| bi. |1-1| njin |2-2| 	a- bein njin 


In [12]:
!paste {file_train_src} {file_train_trg} | grep 'အ ဘိ '

အ ဘိ ဓ ဇ မ ဟာ ရဋ် ဌ ဂု ရု	a- bi. da- za. ma- ha ra' hta. gu. ru.
အ ဘိ ဓမ် မာ	a- bi. da- ma
အ ဘိ ဓာန်	a- bei' dan
အ ဘိ သိက်	a- bei' thei'


In [13]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep 'အဲ့ '

ပုပ် အဲ့ အဲ့	pou' |0-0| အဲ့ |1-1| အဲ့ |2-2| 	pou' e. e. 


In [14]:
!paste {file_train_src} {file_train_trg} | grep 'အဲ့ '

In [15]:
leaderboard[leaderboard.Root_Cause=='Reference Missing from Training Token'].head(30)

,Source_Syllable,Moses_Output,True_Reference,Root_Cause,Error_Count
508,ပျော်,pjo,bjo,Reference Missing from Training Token,2
31,ကုန်,koun,kou,Reference Missing from Training Token,2
605,မြီး,mji:,mhi:,Reference Missing from Training Token,2
567,ဖွင့်,hpwin.,hpwin,Reference Missing from Training Token,1
601,မြ,mja.,mja,Reference Missing from Training Token,1
597,မဲ့,me.,me,Reference Missing from Training Token,1
596,မောင်,maun,moun,Reference Missing from Training Token,1
570,ဖွေ,hpwei,bwei,Reference Missing from Training Token,1
550,ဖည်,hpe,be,Reference Missing from Training Token,1
584,မ,ma-,a',Reference Missing from Training Token,1


In [16]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep 'မြီး'

လိပ် ကျောက် မြီး	lei' |0-0| kyau' |1-1| mji: |2-2| 	lei' kyau' mhi: 
လှည်း နောက် မြီး	lhe: |0-0| nau' |1-1| mji: |2-2| 	lhe: nau' mi: 
ကြွက် မြီး	kywe: mi: |0-1| 	kywe' mi: 
ကိုင်း ဖြူ မြီး	kain: |0-0| hpju |1-1| mji: |2-2| 	kain: bju mi: 
မြီး ထူး ရိုး	mji: |0-0| du: |1-1| jou: |2-2| 	mji: du: jou: 
ဆောင်း မြီး နာ	hsaun: |0-0| mji: |1-1| na |2-2| 	hsaun: mji: na 
လိပ် ကျောက် မြီး	lei' |0-0| kyau' |1-1| mji: |2-2| 	lei' kyau' mi: 
အ မြီး စား ဖက် ခေါင်း စား ဖက်	a- mi: |0-1| sa: |2-2| hpe' |3-3| gaun: |4-4| za: |5-5| hpe' |6-6| 	a- mi: sa: be' gaun: sa: be' 
ကျေး မြီး ဖို	kyei: mi: |0-1| hpou |2-2| 	kyei: mi: bou 
ကင်း မြီး ကောက်	kin: mi: gau' |0-2| 	kin: mi: gau' 
စစ် မြီး	si' |0-0| mji: |1-1| 	si' mhi: 


In [17]:
!paste {file_train_src} {file_train_trg} | grep 'mhi:'

လင် မှီး	lin mhi:
သို မှီး	thou: mhi:
ရှာ မှီး	sha mhi:


In [18]:
leaderboard[leaderboard.Root_Cause=='Decoder Hallucination / Over-generalization'].head(30)

,Source_Syllable,Moses_Output,True_Reference,Root_Cause,Error_Count
414,ဒြေ,da- rei,da-,Decoder Hallucination / Over-generalization,1


In [19]:
!paste {file_train_src} {file_train_trg} | grep  'ဒြေ'

ဣန် ဒြေ ရှိ	ein da- rei shi.
ဣန် ဒြေ လုပ်	ein da- rei lou'
ဣန် ဒြေ မဲ့	ein da- rei me.
ဣန် ဒြေ	ein da- rei
စက် ခုန် ဒြေ	se' khoun da-
ဣန် ဒြေ ဆောင်	ein da- rei hsaun
ဣန် ဒြေ ဆည်	ein da- rei hse
ဣန် ဒြေ သမ် ပတ် တိ	ein da- rei than pa' ti.
ဣန် ဒြေ ပျက်	ein da- rei pje'
ရာ ဇိန် ဒြေ	ja zein da-
ဣန် ဒြေ ရ	ein da- rei ja.


In [20]:
!paste {file_test_src} {file_test_out} {file_test_ref} | grep 'ဒြေ'

ဣန် ဒြေ ကြီး	ein da- rei |0-1| gyi: |2-2| 	ein da- rei kyi: 
ကာ ယိန် ဒြေ	ka |0-0| ယိန် |1-1| da- rei |2-2| 	ka jein da- 
